# 14 — ログと症状からの診断

動画の印象ではなく、因果鎖のどこが先に崩れたかを時系列で見つけます。

**前提**: `13_end_to_end_baseline.ipynb`

> 読み方: 「直感 → 数式 → 上流コード → 小実験 → 解釈」の順です。
> `実装事実` と書いた箇所は現行 `external/Quadruped-PyMPC` のコード、
> `学習用モデル` は理解のために単純化した再実装です。

In [1]:
from pathlib import Path
import os, sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebook_pympc":
    ROOT = ROOT.parent
PYMPC_ROOT = ROOT / "external" / "Quadruped-PyMPC"
assert PYMPC_ROOT.exists(), f"Quadruped-PyMPC が見つかりません: {PYMPC_ROOT}"
if str(PYMPC_ROOT) not in sys.path:
    sys.path.insert(0, str(PYMPC_ROOT))

os.environ.setdefault("ACADOS_SOURCE_DIR", str(PYMPC_ROOT / "quadruped_pympc" / "acados"))
os.environ.setdefault("MUJOCO_GL", "egl")
print("workspace :", ROOT)
print("PyMPC root:", PYMPC_ROOT)

workspace : /home/takuya/work/mpc_dog
PyMPC root: /home/takuya/work/mpc_dog/external/Quadruped-PyMPC


## 最小ログ

- reference / measured: COM位置、速度、roll/pitch/yaw
- gait: phase、current contact、予測contact
- foothold: 離地、参照着地、実着地
- MPC: GRF、予測状態、status、solve time、cost
- low level: Jacobian、raw torque、clip後torque、飽和flag
- Plant: 実接触、実GRF、滑り速度

目標GRFと実GRFを同じ名前で保存しないことが重要です。

In [2]:
import numpy as np
rng = np.random.default_rng(0)
t = np.linspace(0, 4, 401)
ref_vx = np.full_like(t, 0.3)
vx = 0.3*(1-np.exp(-2*t)) + 0.02*rng.normal(size=t.size)
roll = 0.03*np.sin(2*np.pi*1.35*t)
torque = 18 + 8*np.maximum(0, np.sin(2*np.pi*1.35*t))
limit = 21.33

metrics = {
    "vx_rmse": np.sqrt(np.mean((vx-ref_vx)**2)),
    "roll_rms_rad": np.sqrt(np.mean(roll**2)),
    "torque_saturation_rate": np.mean(torque >= limit),
}
metrics

{'vx_rmse': np.float64(0.07794550867930047),
 'roll_rms_rad': np.float64(0.02134342006883563),
 'torque_saturation_rate': np.float64(0.39650872817955113)}

## 診断順

1. solver status/NaN
2. contact予測と実接触のずれ
3. 摩擦marginとGRFの急変
4. torque飽和
5. 姿勢・速度誤差
6. 転倒

最後に見えた転倒ではなく、最初に閾値を越えた内部量を原因候補にします。

In [3]:
thresholds = {"vx_rmse": 0.12, "roll_rms_rad": 0.08, "torque_saturation_rate": 0.05}
for key, value in metrics.items():
    print(f"{key:26s} {value:.4f}  {'NG' if value > thresholds[key] else 'OK'}")

vx_rmse                    0.0779  OK
roll_rms_rad               0.0213  OK
torque_saturation_rate     0.3965  NG


## 章末チェック

出力を眺めるだけでなく、次を自分の言葉で答えてください。

1. この章の入力・出力の shape、単位、座標系は何か。
2. 変更可能な量と、他の章から渡される量は何か。
3. パラメータを2倍にしたとき、どのグラフがどちらへ変化するか。
4. 現行実装の事実と、学習用の近似を区別できるか。